# PlacementIQ — Notebook 02: Exploratory Data Analysis

Objective: Understand feature distributions, relationships with the target (`placement_status`), correlations, outliers, and synthetic data patterns.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13, 'axes.labelsize': 11})

FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset
DATA_PATH = Path('../data/raw/student_placement_synthetic.csv')
df_raw = pd.read_csv(DATA_PATH)

# Drop salary_package_lpa — target leakage
df = df_raw.drop(columns=['salary_package_lpa'])

NUMERICAL_FEATURES = [
    'cgpa', 'backlogs', 'coding_skills', 'dsa_score', 'aptitude_score',
    'communication_skills', 'ml_knowledge', 'system_design', 'internships',
    'projects_count', 'certifications', 'hackathons',
    'open_source_contributions', 'extracurriculars'
]
CATEGORICAL_FEATURES = ['branch', 'college_tier']
TARGET = 'placement_status'

print(f'Dataset shape after dropping salary_package_lpa: {df.shape}')
df.head(3)

## 2. Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

counts = df[TARGET].value_counts()
pcts = df[TARGET].value_counts(normalize=True) * 100
labels_map = {0: 'Not Placed (0)', 1: 'Placed (1)'}
bar_colors = ['#F44336', '#4CAF50']

bars = ax.bar(
    [labels_map[k] for k in counts.index],
    counts.values,
    color=bar_colors,
    edgecolor='white',
    linewidth=1.5
)
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 400,
        f'{cnt:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

ax.set_title('Target Variable Distribution: placement_status', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.2)
sns.despine()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Class imbalance ratio (Placed:NotPlaced): {counts[1]/counts[0]:.2f}:1')

## 3. Numerical Distributions

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(NUMERICAL_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(NUMERICAL_FEATURES):
    ax = axes[i]
    ax.hist(df[feat], bins=30, color='steelblue', edgecolor='white', linewidth=0.5, alpha=0.85)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    # Annotate mean & median
    mean_val = df[feat].mean()
    med_val = df[feat].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.2, label=f'Mean={mean_val:.2f}')
    ax.axvline(med_val, color='orange', linestyle='-', linewidth=1.2, label=f'Median={med_val:.2f}')
    ax.legend(fontsize=7)

# Hide extra subplots
for j in range(len(NUMERICAL_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Box Plots by Placement Status

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(NUMERICAL_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

palette = {0: '#F44336', 1: '#4CAF50'}

for i, feat in enumerate(NUMERICAL_FEATURES):
    ax = axes[i]
    sns.boxplot(
        data=df, x=TARGET, y=feat, ax=ax,
        hue=TARGET, palette=palette, legend=False,
        width=0.5, linewidth=1.2, fliersize=2
    )
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('placement_status')
    ax.set_xticklabels(['Not Placed (0)', 'Placed (1)'])

for j in range(len(NUMERICAL_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Box Plots: Numerical Features by Placement Status', fontsize=16,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_boxplots_by_placement.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
CORR_FEATURES = NUMERICAL_FEATURES + [TARGET]
corr_matrix = df[CORR_FEATURES].corr(method='pearson')

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Pearson Correlation Heatmap (Numerical Features + Target)',
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr_matrix[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print('Top correlations with placement_status:')
print(target_corr.round(4).to_string())

## 6. Categorical Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = {0: '#F44336', 1: '#4CAF50'}

# Branch by placement_status
branch_order = df['branch'].value_counts().index.tolist()
sns.countplot(
    data=df, x='branch', hue=TARGET, ax=axes[0],
    order=branch_order, palette=palette
)
axes[0].set_title('Branch vs Placement Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Branch')
axes[0].set_ylabel('Count')
axes[0].legend(title='placement_status', labels=['Not Placed', 'Placed'])

# College Tier by placement_status
tier_order = ['Tier-1', 'Tier-2', 'Tier-3']
sns.countplot(
    data=df, x='college_tier', hue=TARGET, ax=axes[1],
    order=tier_order, palette=palette
)
axes[1].set_title('College Tier vs Placement Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('College Tier')
axes[1].set_ylabel('Count')
axes[1].legend(title='placement_status', labels=['Not Placed', 'Placed'])

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Placement rates by branch
branch_rates = df.groupby('branch')[TARGET].agg(['sum', 'count'])
branch_rates['placement_rate_%'] = (branch_rates['sum'] / branch_rates['count'] * 100).round(2)
branch_rates.columns = ['Placed', 'Total', 'Placement Rate (%)']
branch_rates = branch_rates.sort_values('Placement Rate (%)', ascending=False)
print('=== Placement Rate by Branch ===')
print(branch_rates.to_string())

# Placement rates by college_tier
tier_rates = df.groupby('college_tier')[TARGET].agg(['sum', 'count'])
tier_rates['placement_rate_%'] = (tier_rates['sum'] / tier_rates['count'] * 100).round(2)
tier_rates.columns = ['Placed', 'Total', 'Placement Rate (%)']
tier_rates = tier_rates.sort_values('Placement Rate (%)', ascending=False)
print('\n=== Placement Rate by College Tier ===')
print(tier_rates.to_string())

## 7. Feature vs Target Analysis

In [ ]:
# Mean value per feature by placement_status
mean_by_target = df.groupby(TARGET)[NUMERICAL_FEATURES].mean()
mean_diff = (mean_by_target.loc[1] - mean_by_target.loc[0]).abs().sort_values(ascending=False)

print('=== Absolute Mean Difference (Placed - Not Placed) ===')
print(mean_diff.round(4).to_string())

# Bar chart of mean values per feature by placement_status
n_cols = 4
n_rows = int(np.ceil(len(NUMERICAL_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()
palette_target = ['#F44336', '#4CAF50']

for i, feat in enumerate(mean_diff.index):  # ordered by diff
    ax = axes[i]
    group_means = mean_by_target[feat]
    bars = ax.bar(
        ['Not Placed (0)', 'Placed (1)'],
        group_means.values,
        color=palette_target,
        edgecolor='white', linewidth=1.2
    )
    for bar, val in zip(bars, group_means.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'{feat}\n(Δ={mean_diff[feat]:.3f})', fontsize=10, fontweight='bold')
    ax.set_ylabel('Mean Value')

for j in range(len(NUMERICAL_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Mean Feature Values by Placement Status (sorted by |Δ|)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_feature_vs_target_means.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTop discriminating features by mean difference:')
print(mean_diff.head(5).round(3).to_string())

## 8. Outlier Investigation

In [ ]:
print('=== IQR-Based Outlier Detection ===')
outlier_summary = {}
for feat in NUMERICAL_FEATURES:
    Q1 = df[feat].quantile(0.25)
    Q3 = df[feat].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = int(((df[feat] < lower) | (df[feat] > upper)).sum())
    pct = n_outliers / len(df) * 100
    outlier_summary[feat] = n_outliers
    print(f'  {feat:30s}: {n_outliers:5d} outliers ({pct:.2f}%)')

total_outlier_rows = sum(outlier_summary.values())
print(f'\nTotal outlier-flagged instances (sum across features): {total_outlier_rows:,}')
print('\nNote: Tree-based models (XGBoost, Random Forest) are robust to outliers.')
print('Removal is NOT recommended for this project.')

## 9. Pairwise Relationships

In [ ]:
PAIRPLOT_FEATURES = ['cgpa', 'dsa_score', 'coding_skills', 'aptitude_score', 'internships', TARGET]
pairplot_df = df[PAIRPLOT_FEATURES].copy()
pairplot_df[TARGET] = pairplot_df[TARGET].map({0: 'Not Placed', 1: 'Placed'})

print('Generating pairplot — this may take a moment on 100K rows...')
# Sample 10K for pairplot performance
sample_df = pairplot_df.sample(n=10_000, random_state=42)

g = sns.pairplot(
    sample_df,
    hue=TARGET,
    palette={'Not Placed': '#F44336', 'Placed': '#4CAF50'},
    plot_kws={'alpha': 0.3, 's': 10},
    diag_kind='kde'
)
g.fig.suptitle(
    'Pairwise Relationships — Selected Features (10K sample)',
    y=1.02, fontsize=14, fontweight='bold'
)
plt.savefig(FIGURES_DIR / '02_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()
print('Pairplot saved.')

## 10. Synthetic Data Pattern Check

> **⚠️ Warning — Synthetic Dataset**: This dataset is synthetically generated. Extremely high point-biserial correlations or mutual information scores may indicate that the target was generated via deterministic rules (thresholds) rather than noisy real-world processes. Such patterns would not generalise to production data.

In [ ]:
from scipy import stats
from sklearn.feature_selection import mutual_info_classif

# Point-biserial correlations
print('=== Point-Biserial Correlation with placement_status ===')
pb_results = {}
for feat in NUMERICAL_FEATURES:
    corr, pval = stats.pointbiserialr(df[feat], df[TARGET])
    pb_results[feat] = {'pb_corr': round(corr, 4), 'p_value': round(pval, 6)}

pb_df = pd.DataFrame(pb_results).T
pb_df = pb_df.sort_values('pb_corr', key=abs, ascending=False)
print(pb_df.to_string())

# Flag suspiciously high values
HIGH_THRESHOLD = 0.5
high_corr = pb_df[pb_df['pb_corr'].abs() > HIGH_THRESHOLD]
if not high_corr.empty:
    print(f'\n⚠️  Features with |pb_corr| > {HIGH_THRESHOLD} (possibly rule-based):')
    print(high_corr.to_string())
else:
    print(f'\nNo features with |pb_corr| > {HIGH_THRESHOLD}.')

# Mutual Information scores
print('\n=== Mutual Information Scores with placement_status ===')
X_num = df[NUMERICAL_FEATURES].values
y = df[TARGET].values
mi_scores = mutual_info_classif(X_num, y, random_state=42)
mi_df = pd.Series(mi_scores, index=NUMERICAL_FEATURES, name='mutual_info').sort_values(ascending=False)
print(mi_df.round(4).to_string())

# Bar chart of MI scores
fig, ax = plt.subplots(figsize=(10, 5))
mi_df.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Mutual Information Scores vs placement_status', fontsize=13, fontweight='bold')
ax.set_xlabel('Mutual Information')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. EDA Summary

### Key Findings

**Target Variable**
- 68,475 Placed (68.5%) vs 31,525 Not Placed (31.5%) — moderate class imbalance requiring SMOTE or class-weight adjustments during modelling.

**Numerical Distributions**
- `cgpa` is roughly bell-shaped between 4–10; placed students have noticeably higher CGPA.
- `aptitude_score` spans 20–100 with clear separation between placed and not-placed.
- Count features (`internships`, `projects_count`, `certifications`, `hackathons`, `open_source_contributions`, `extracurriculars`) are discrete and right-skewed.
- `backlogs` is inversely associated with placement — students with 0 backlogs are more likely to be placed.

**Correlations**
- Top features correlated with `placement_status`: `cgpa`, `dsa_score`, `coding_skills`, `aptitude_score` (see heatmap).
- Inter-feature correlations are generally low-to-moderate, suggesting limited multicollinearity.

**Categorical Features**
- CSE and IT branches tend to have higher placement rates than Chemical or ME.
- Tier-1 colleges show consistently higher placement rates than Tier-2 and Tier-3.

**Outliers**
- IQR analysis reveals some outliers in continuous features, but counts are low.
- **No removal recommended** — XGBoost and tree-based ensembles are robust to outliers.

**Synthetic Data Patterns**
- Point-biserial correlations and mutual information scores reveal the strength of each feature's relationship with the target.
- Any unusually high values (|pb_corr| > 0.5) would suggest deterministic rule-based generation, warranting caution when interpreting model performance.

**Next Steps → Notebook 03: Preprocessing & Feature Engineering**
- One-hot encode `branch` and `college_tier`.
- Scale numerical features with StandardScaler (fitted on train only).
- Apply SMOTE on training fold to address class imbalance.
- Proceed to 5-fold stratified cross-validation in Notebook 04.